# Join CL lower court Data to SCOTUS Scraped data

The SCOTUS Scraped data is used as a bridge to link the CL SCOTUS opinions to lower court opinions that appealed to SCOTUS. In order to establish the appellate linkage between SCOTUS and lower court opinions, we will leverage the docket number and the date to
1. Link the CL SCOTUS opinions to SCOTUS scraped data
2. Link the CL Lower Court opinions to SCOTUS scraped data

This notebook outlines the steps undertook to map the SCOTUS lower court names to CL court names in preparation for step 2 above.

# Import Libaries

In [1]:
import numpy as np
import pandas as pd

from courts_db import find_court, find_court_by_id

# Load the SCOTUS scraped data

In [2]:
scotus = pd.read_json("data/scotus_scraped_data.json")
scotus.head()

,docket_number,filename,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
0,03-6084,03-6084.htm,2003-08-27,"Ronald Lee Smith, Petitioner v. Larry Reid, Wa...",United States Court of Appeals for the Tenth C...,(03-1016),03-1016,2003-05-16,2003-07-09,3,6084
1,03-10616,03-10616.htm,2004-05-28,"Jimmy Walker, Petitioner v. Florida","District Court of Appeal of Florida, Fourth Di...",(4D02-4272),4D02-4272,2004-04-21,None,3,10616
2,03-777,03-777.htm,2003-11-28,"Willie R. Flint, Petitioner v. ABB Inc., fka A...",United States Court of Appeals for the Elevent...,(02-15029),02-15029,2003-07-21,2003-08-27,3,777
3,03-10170,03-10170.htm,2004-05-05,"Loyda Lugones, Petitioner v. United States",United States Court of Appeals for the Elevent...,(02-12984),02-12984,2004-01-26,None,3,10170
4,03-8917,03-8917.htm,2004-02-19,"Ray Charles Smith, Petitioner v. United States",United States Court of Appeals for the Ninth C...,(03-10003),03-10003,2003-11-13,None,3,8917


In [3]:
scotus.columns

Index(['docket_number', 'filename', 'docket_date', 'case_title', 'lower_court',
       'lower_court_case_numbers_raw', 'lower_court_case_numbers',
       'lower_court_decision_date', 'lower_court_rehearing_denied_date',
       'year', 'case_num'],
      dtype='object')

# Use courts-db to map the lower court names to cl court ids and cl court names

In [5]:
courts_summary = (
    scotus.groupby("lower_court")
      .agg(count=("lower_court", "size"), first_case_num=("lower_court_case_numbers", "first"))
      .reset_index()
      .sort_values("count", ascending=False)
)

courts_summary.head(10)

,lower_court,count,first_case_num
603,United States Court of Appeals for the Ninth C...,24127,03-10003
600,United States Court of Appeals for the Fifth C...,20633,01-41164
598,United States Court of Appeals for the Elevent...,17573,02-15029
602,United States Court of Appeals for the Fourth ...,15380,03-1569
606,United States Court of Appeals for the Sixth C...,12881,03-2095
608,United States Court of Appeals for the Third C...,9470,02-2497
604,United States Court of Appeals for the Second ...,9219,02-1110
597,United States Court of Appeals for the Eighth ...,8893,02-4108
605,United States Court of Appeals for the Seventh...,8028,02-3595
607,United States Court of Appeals for the Tenth C...,6876,03-1016


In [6]:
courts_summary["cl_court_ids"] = courts_summary["lower_court"].apply(lambda x: find_court(x, bankruptcy=False))
courts_summary["cl_court_ids_len"] = courts_summary["cl_court_ids"].apply(len)

courts_summary.head(5)

,lower_court,count,first_case_num,cl_court_ids,cl_court_ids_len
603,United States Court of Appeals for the Ninth C...,24127,03-10003,[ca9],1
600,United States Court of Appeals for the Fifth C...,20633,01-41164,[ca5],1
598,United States Court of Appeals for the Elevent...,17573,02-15029,[ca11],1
602,United States Court of Appeals for the Fourth ...,15380,03-1569,[ca4],1
606,United States Court of Appeals for the Sixth C...,12881,03-2095,[ca6],1


In [7]:
courts_summary["cl_court_ids_len"].value_counts()

cl_court_ids_len
0    477
1    200
Name: count, dtype: int64

In [8]:
courts_summary["cl_court_id"] = courts_summary["cl_court_ids"].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else np.nan)
courts_summary.drop('cl_court_ids', axis=1, inplace=True)
courts_summary.drop('cl_court_ids_len', axis=1, inplace=True)
courts_summary.head()

,lower_court,count,first_case_num,cl_court_id
603,United States Court of Appeals for the Ninth C...,24127,03-10003,ca9
600,United States Court of Appeals for the Fifth C...,20633,01-41164,ca5
598,United States Court of Appeals for the Elevent...,17573,02-15029,ca11
602,United States Court of Appeals for the Fourth ...,15380,03-1569,ca4
606,United States Court of Appeals for the Sixth C...,12881,03-2095,ca6


In [9]:
courts_summary["cl_court_name"] = courts_summary["cl_court_id"].apply(lambda x: find_court_by_id(x)[0]["name"] if find_court_by_id(x) else None)
courts_summary.head()

,lower_court,count,first_case_num,cl_court_id,cl_court_name
603,United States Court of Appeals for the Ninth C...,24127,03-10003,ca9,Court of Appeals for the Ninth Circuit
600,United States Court of Appeals for the Fifth C...,20633,01-41164,ca5,Court of Appeals for the Fifth Circuit
598,United States Court of Appeals for the Elevent...,17573,02-15029,ca11,United States Court of Appeals for the Elevent...
602,United States Court of Appeals for the Fourth ...,15380,03-1569,ca4,Court of Appeals for the Fourth Circuit
606,United States Court of Appeals for the Sixth C...,12881,03-2095,ca6,Court of Appeals for the Sixth Circuit


In [10]:
courts_summary.to_csv("data/scotus_lower_courts.csv")

# Manually map the remaining courts

Only ~1/3 of the lower courts were mapped to cl courts, the remaining requires manual mapping. I mapped them manually with the exception of local (city, county, parish, etc) level trial courts, where we either do not have them in the court database or it's unclear which court id to assign to. These records can be discarded for appellate chain linking purpose as we do not have the trial court level opinions for most trial courts.

In [2]:
courts_summary = pd.read_csv("data/scotus_lower_courts_mapped.csv")
courts_summary.head()

,lower_court,count,first_case_num,cl_court_id,cl_court_name,manual,manual_type,auto_type,type
0,United States Court of Appeals for the Ninth C...,24127,03-10003,ca9,Court of Appeals for the Ninth Circuit,0,NaN,Federal Appellate,Federal Appellate
1,United States Court of Appeals for the Fifth C...,20633,01-41164,ca5,Court of Appeals for the Fifth Circuit,0,NaN,Federal Appellate,Federal Appellate
2,United States Court of Appeals for the Elevent...,17573,02-15029,ca11,United States Court of Appeals for the Elevent...,0,NaN,Federal Appellate,Federal Appellate
3,United States Court of Appeals for the Fourth ...,15380,03-1569,ca4,Court of Appeals for the Fourth Circuit,0,NaN,Federal Appellate,Federal Appellate
4,United States Court of Appeals for the Sixth C...,12881,03-2095,ca6,Court of Appeals for the Sixth Circuit,0,NaN,Federal Appellate,Federal Appellate


In [3]:
len(courts_summary)

677

In [4]:
courts_summary["manual"].value_counts()

manual
1    479
0    198
Name: count, dtype: int64

In [5]:
courts_summary["type"].value_counts()

type
State Trial                 419
State Appellate             111
Federal District             67
State Supreme                58
Federal Appellate            13
Territory Supreme             4
Military Appellate            1
State Special                 1
Territory Appellate           1
Federal Bankruptcy Panel      1
Federal Special               1
Name: count, dtype: int64

In [10]:
len(courts_summary[~courts_summary["cl_court_id"].isnull()])

268

In [6]:
courts_summary[~courts_summary["cl_court_id"].isnull()]["type"].value_counts().sort_index()

type
Federal Appellate            13
Federal Bankruptcy Panel      1
Federal District             66
Federal Special               1
Military Appellate            1
State Appellate             110
State Special                 1
State Supreme                58
State Trial                  12
Territory Appellate           1
Territory Supreme             4
Name: count, dtype: int64

In [7]:
courts_summary[courts_summary["cl_court_id"].isnull()]["type"].value_counts().sort_index()

type
Federal District      1
State Appellate       1
State Trial         407
Name: count, dtype: int64

In [9]:
courts_summary[(courts_summary["cl_court_id"].isnull()) & (courts_summary["type"] != "State Trial")]

,lower_court,count,first_case_num,cl_court_id,cl_court_name,manual,manual_type,auto_type,type
207,United States District Courts for the Eastern ...,11,"CIV S-90-0520, C01-1351",NaN,NaN,1,NaN,NaN,Federal District
208,"District Court of Appeals of Florida, Sixth Di...",10,"6D2024-0641, 6D2024-0147",NaN,NaN,1,NaN,NaN,State Appellate


Of the ones without cl_court_id mapping, one is Federal District (for California), another is State Appellate (For Florida). The California one is because the lower_court provided by scotus encompass two districts, whereas in our DB, the two districts are represented by two ids. The Florida one is because the lower_court is missing in DB, which we will add to our DB https://github.com/freelawproject/courts-db/issues/114. The remaining are State Trial courts, which is outside the scope of the current excercise.